# ClinAuthBench v1 Baseline Notebook

This notebook demonstrates how to load ClinAuthBench v1 from Hugging Face, inspect a case, build a zero-shot prompt, and compute first-pass metrics.

It is intentionally lightweight. The goal is usability and benchmark protocol, not a full leaderboard or production evaluation harness.

## 1. Install And Import

If you are running in Colab, uncomment the install line.

In [ ]:
# %pip install -q datasets

import json
import re
from statistics import mean

from datasets import load_dataset

## 2. Load ClinAuthBench From Hugging Face

Edit `DATASET_ID` if you publish or mirror the dataset under a different Hugging Face namespace.

In [ ]:
DATASET_ID = "Shivi1982/clin-auth-bench"

ds = load_dataset(DATASET_ID, split="test")
ds

## 3. Inspect One Case

Each record has `id`, `title`, `content`, and `metadata`. The model-facing chart packet is in `content`; gold labels are in `metadata.gold`.

In [ ]:
case = ds[0]

print("Case ID:", case["id"])
print("Title:", case["title"])
print("\nFirst 1000 characters of content:\n")
print(case["content"][:1000])
print("\nGold metadata:\n")
print(json.dumps(case["metadata"]["gold"], indent=2))

## 4. Build A Zero-Shot Prompt

The prompt asks a model to return strict JSON for a small set of benchmark fields. It also includes the case-specific `do_not_claim` rules so unsupported-claim behavior can be evaluated.

In [ ]:
def build_zero_shot_prompt(record):
    gold = record["metadata"]["gold"]
    do_not_claim = gold.get("do_not_claim", [])
    do_not_claim_text = "\n".join(f"- {rule}" for rule in do_not_claim)

    return f"""
You are evaluating a synthetic inpatient health authorization chart packet.

Task: read the chart packet and return ONLY valid JSON with this schema:
{{
  "safe_for_lloc": true,
  "expected_los_recommendation": "0 days | 1 day | 2 days | 3 days | other",
  "current_suicide_risk": "short evidence-grounded label",
  "evidence_forms": ["form names that support the answer"],
  "claims_to_avoid": ["unsupported claims the model avoided"],
  "rationale": "brief evidence-grounded explanation"
}}

Do not invent evidence. Do not rely on one favorable note if the broader packet contradicts it.

Case-specific unsupported claims to avoid:
{do_not_claim_text}

CHART PACKET:
{record["content"]}
""".strip()


prompt = build_zero_shot_prompt(ds[0])
print(prompt[:2500])

## 5. Run 3-5 Sample Cases

This cell prepares a few sample prompts. Use them for manual copy/paste testing, or connect your preferred model API in the placeholder function below.

In [ ]:
SAMPLE_INDICES = [0, 120, 170]
sample_records = [ds[i] for i in SAMPLE_INDICES]

for record in sample_records:
    print("=" * 80)
    print(record["id"], "|", record["title"])
    print(build_zero_shot_prompt(record)[:1200])
    print()

In [ ]:
def call_model_placeholder(prompt):
    """Replace this function with your preferred model API call.

    Expected return type: a Python dict matching the JSON schema in the prompt.
    Keep this notebook provider-neutral for the first public baseline.
    """
    raise NotImplementedError("Add your model call here.")


# Example usage after implementing call_model_placeholder:
# predictions = []
# for record in sample_records:
#     prompt = build_zero_shot_prompt(record)
#     pred = call_model_placeholder(prompt)
#     pred["case_id"] = record["id"]
#     predictions.append(pred)

## 6. Metric Helpers

These are first-pass metric helpers:

- `safe_for_lloc` accuracy
- `expected_los_recommendation` exact match
- evidence-anchor form recall
- heuristic `do_not_claim` violation count

The `do_not_claim` helper is only a lexical proxy. Full unsupported-claim evaluation should use human review or a stronger semantic judge.

In [ ]:
def normalize_text(value):
    return re.sub(r"\s+", " ", str(value or "").strip().lower())


def safe_for_lloc_accuracy(records, predictions):
    correct = 0
    total = 0
    for record, pred in zip(records, predictions):
        if "safe_for_lloc" not in pred:
            continue
        gold = bool(record["metadata"]["gold"]["safe_for_lloc"])
        predicted = bool(pred["safe_for_lloc"])
        correct += int(gold == predicted)
        total += 1
    return correct / total if total else None


def expected_los_exact_match(records, predictions):
    correct = 0
    total = 0
    for record, pred in zip(records, predictions):
        if "expected_los_recommendation" not in pred:
            continue
        gold = normalize_text(record["metadata"]["gold"]["expected_los_recommendation"])
        predicted = normalize_text(pred["expected_los_recommendation"])
        correct += int(gold == predicted)
        total += 1
    return correct / total if total else None


def gold_evidence_forms(record):
    gold = record["metadata"]["gold"]
    forms = set(gold.get("key_evidence_forms", []))
    for anchor in gold.get("evidence_anchors", []):
        supporting_form = anchor.get("supporting_form")
        if supporting_form:
            forms.add(supporting_form)
    return forms


def evidence_anchor_form_recall(records, predictions):
    recalls = []
    for record, pred in zip(records, predictions):
        gold_forms = {normalize_text(form) for form in gold_evidence_forms(record)}
        pred_forms = {normalize_text(form) for form in pred.get("evidence_forms", [])}
        if not gold_forms:
            continue
        recalls.append(len(gold_forms & pred_forms) / len(gold_forms))
    return mean(recalls) if recalls else None


STOPWORDS = {
    "a", "an", "and", "are", "as", "be", "but", "by", "do", "does", "from", "in",
    "is", "it", "not", "of", "or", "that", "the", "to", "with", "without"
}


def important_terms(text):
    terms = re.findall(r"[a-zA-Z][a-zA-Z-]+", normalize_text(text))
    return {term for term in terms if len(term) >= 4 and term not in STOPWORDS}


def do_not_claim_violation_count(records, predictions, min_overlap=3):
    """Lexical proxy for unsupported claims.

    This checks only narrative fields where the model makes claims. It intentionally
    ignores `claims_to_avoid`, because that field may repeat the rules correctly.
    """
    violations = []
    for record, pred in zip(records, predictions):
        gold_rules = record["metadata"]["gold"].get("do_not_claim", [])
        model_claim_text = " ".join(
            str(pred.get(field, ""))
            for field in ["rationale", "authorization_summary", "predicted_claims", "current_suicide_risk"]
        )
        model_terms = important_terms(model_claim_text)
        for rule in gold_rules:
            overlap = important_terms(rule) & model_terms
            if len(overlap) >= min_overlap:
                violations.append({
                    "case_id": record["id"],
                    "rule": rule,
                    "overlap_terms": sorted(overlap),
                })
    return violations


def evaluate_predictions(records, predictions):
    violations = do_not_claim_violation_count(records, predictions)
    return {
        "n": len(predictions),
        "safe_for_lloc_accuracy": safe_for_lloc_accuracy(records, predictions),
        "expected_los_exact_match": expected_los_exact_match(records, predictions),
        "evidence_anchor_form_recall": evidence_anchor_form_recall(records, predictions),
        "do_not_claim_violation_count_heuristic": len(violations),
        "do_not_claim_violation_examples": violations[:5],
    }

## 7. Test Metrics With Mock Predictions

The predictions below are mock outputs to test the metric code. They are not model results and should not be reported as benchmark performance.

In [ ]:
mock_predictions = []

for record in sample_records:
    gold = record["metadata"]["gold"]
    mock_predictions.append({
        "case_id": record["id"],
        "safe_for_lloc": gold["safe_for_lloc"],
        "expected_los_recommendation": gold["expected_los_recommendation"],
        "current_suicide_risk": gold["current_suicide_risk"],
        "evidence_forms": gold.get("key_evidence_forms", [])[:3],
        "claims_to_avoid": gold.get("do_not_claim", []),
        "rationale": "Mock prediction for metric testing only.",
    })

metrics = evaluate_predictions(sample_records, mock_predictions)
print(json.dumps(metrics, indent=2))

## 8. Suggested Next Steps

- Replace `call_model_placeholder` with a real model call.
- Run the prompt on all 180 cases.
- Report sample size, model, prompt version, and parsing failure rate.
- Treat the heuristic `do_not_claim` check as a screening signal, not a final hallucination judge.
- Use human or LLM adjudication for semantic unsupported-claim evaluation.